In [1]:
# Cell 1 — Imports, paths, and evaluation settings
#
# Purpose:
# Measure manual-trace coverage within the P28 ROI and compare
# the three correction methods using identical filtering settings.

from pathlib import Path
import gzip
import json
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

from scipy import ndimage as ndi
from skimage.draw import line_nd


# --------------------------------------------------
# 1. Project paths
# --------------------------------------------------

def find_project_root(start=None):
    start_path = Path.cwd() if start is None else Path(start)
    start_path = start_path.resolve()

    for directory in (start_path, *start_path.parents):
        if (
            (directory / "README.md").is_file()
            and (directory / "notebooks").is_dir()
            and (directory / "data").is_dir()
        ):
            return directory

    raise FileNotFoundError(
        "Run this notebook from within the repository containing "
        "README.md, notebooks/, and data/."
    )


PROJECT_ROOT = find_project_root()


def project_relative(path):
    return Path(path).relative_to(PROJECT_ROOT).as_posix()

SAMPLE_NAME = "p28_330_HET_009"

RECONSTRUCTION_DIR = (
    PROJECT_ROOT
    / "results"
    / SAMPLE_NAME
    / "07_reconstruction"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / SAMPLE_NAME
    / "08_evaluation"
)

run_info_file = RECONSTRUCTION_DIR / "run_info.json"

if not run_info_file.is_file():
    raise FileNotFoundError(
        f"Run record not found: {project_relative(run_info_file)}"
    )

run_info = json.loads(
    run_info_file.read_text(encoding="utf-8")
)


# --------------------------------------------------
# 2. Check the reconstruction record
# --------------------------------------------------

if run_info["sample"] != SAMPLE_NAME:
    raise ValueError("The saved sample does not match SAMPLE_NAME.")

if run_info["array_order_3d"] != "Z,Y,X":
    raise ValueError("Expected saved arrays in Z,Y,X order.")

if not run_info["solver"]["converged"]:
    raise RuntimeError("The saved hitting map did not converge.")


# --------------------------------------------------
# 3. Locate the evaluation inputs
# --------------------------------------------------

required_inputs = [
    "hitting_probability",
    "path_length",
    "adjusted_score",
    "active_domain",
    "soma_target",
    "normalized_raw_mip",
    "final_mask",
    "roi_metadata",
]

INPUT_FILES = {
    name: RECONSTRUCTION_DIR / run_info["files"][name]
    for name in required_inputs
}

TRACE_FILE = PROJECT_ROOT / run_info["manual_trace_file"]

missing_files = [
    project_relative(path)
    for path in [*INPUT_FILES.values(), TRACE_FILE]
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing evaluation inputs:\n" + "\n".join(missing_files)
    )


# --------------------------------------------------
# 4. Read the actual reconstruction settings
# --------------------------------------------------

SCORE_THRESHOLD = float(
    run_info["intensity_filter"]["score_threshold"]
)

INTENSITY_THRESHOLD = float(
    run_info["intensity_filter"]["intensity_threshold"]
)

APPLY_COMPONENT_FILTER = bool(
    run_info["mask"]["component_filtering"]
)

MIN_COMPONENT_VOXELS = (
    int(run_info["mask"]["minimum_component_voxels"])
    if APPLY_COMPONENT_FILTER
    else None
)

CONNECTIVITY_3D = np.ones((3, 3, 3), dtype=bool)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample: {SAMPLE_NAME}")
print(f"Reconstruction directory: {project_relative(RECONSTRUCTION_DIR)}")
print(f"Manual tracing: {project_relative(TRACE_FILE)}")
print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Raw-MIP intensity threshold: {INTENSITY_THRESHOLD}")
print(f"Component filtering: {APPLY_COMPONENT_FILTER}")

if APPLY_COMPONENT_FILTER:
    print(f"Minimum component size: {MIN_COMPONENT_VOXELS} voxels")

print(f"Evaluation output directory: {project_relative(OUTPUT_DIR)}")

Sample: p28_330_HET_009
Reconstruction directory: results/p28_330_HET_009/07_reconstruction
Manual tracing: data/p28_330_HET_009/manual_tracing/p28_330_HET_009_manual.traces
Score threshold: 0.5
Raw-MIP intensity threshold: 0.12
Component filtering: True
Minimum component size: 10 voxels
Evaluation output directory: results/p28_330_HET_009/08_evaluation


In [2]:
# Cell 2 — Load the saved reconstruction arrays and ROI metadata

# --------------------------------------------------
# 1. Read the ROI metadata
# --------------------------------------------------

metadata_table = pd.read_csv(INPUT_FILES["roi_metadata"])

if len(metadata_table) != 1:
    raise ValueError("Expected one row of ROI metadata.")

metadata = metadata_table.iloc[0]

shape_zyx = tuple(
    int(n) for n in run_info["shape_zyx"]
)

metadata_shape_zyx = (
    int(metadata["dim_z"]),
    int(metadata["dim_y"]),
    int(metadata["dim_x"]),
)

if metadata_shape_zyx != shape_zyx:
    raise ValueError("ROI metadata and saved array dimensions differ.")

roi_x0 = int(metadata["global_x0_zero_based"])
roi_y0 = int(metadata["global_y0_zero_based"])


# --------------------------------------------------
# 2. Open the saved arrays in read-only mode
# --------------------------------------------------

saved_arrays = {
    name: np.load(
        path,
        mmap_mode="r",
        allow_pickle=False,
    )
    for name, path in INPUT_FILES.items()
    if name != "roi_metadata"
}

P_hit_zyx = saved_arrays["hitting_probability"]
L_zyx = saved_arrays["path_length"]
sqrtL_score_zyx = saved_arrays["adjusted_score"]

active_zyx = saved_arrays["active_domain"]
target_zyx = saved_arrays["soma_target"]
final_mask_zyx = saved_arrays["final_mask"]

normalized_intensity_yx = saved_arrays["normalized_raw_mip"]


# --------------------------------------------------
# 3. Verify shapes and mask types
# --------------------------------------------------

for name, array in saved_arrays.items():
    expected_shape = (
        shape_zyx[1:]
        if name == "normalized_raw_mip"
        else shape_zyx
    )

    if array.shape != expected_shape:
        raise ValueError(
            f"{name}: expected shape {expected_shape}, "
            f"found {array.shape}."
        )

for name in ("active_domain", "soma_target", "final_mask"):
    if saved_arrays[name].dtype != np.dtype(bool):
        raise ValueError(f"Expected a Boolean mask for {name}.")

assert target_zyx.any(), "The soma target is empty."
assert np.all(active_zyx[target_zyx])
assert np.all(final_mask_zyx[target_zyx])
assert not np.any(final_mask_zyx & ~active_zyx)

assert np.isfinite(P_hit_zyx).all()
assert np.isfinite(sqrtL_score_zyx).all()
assert np.isfinite(normalized_intensity_yx).all()

# Infinite path lengths are valid for unreachable voxels.
assert not np.isnan(L_zyx).any()


# --------------------------------------------------
# 4. Report the loaded data
# --------------------------------------------------

print(f"Volume shape (Z, Y, X): {shape_zyx}")
print(f"ROI origin (global X, Y): ({roi_x0}, {roi_y0})")
print(f"Active voxels: {int(active_zyx.sum()):,}")
print(f"Soma-target voxels: {int(target_zyx.sum()):,}")
print(f"Final-mask voxels: {int(final_mask_zyx.sum()):,}")
print("Saved arrays loaded in read-only mode.")

Volume shape (Z, Y, X): (161, 241, 241)
ROI origin (global X, Y): (742, 433)
Active voxels: 1,606,031
Soma-target voxels: 2,829
Final-mask voxels: 233,435
Saved arrays loaded in read-only mode.


In [3]:
# Cell 3 — Read and rasterize the manual tracing

# --------------------------------------------------
# 1. Read the tracing XML
# --------------------------------------------------

with TRACE_FILE.open("rb") as file:
    is_gzipped = file.read(2) == b"\x1f\x8b"

trace_open = gzip.open if is_gzipped else open

with trace_open(TRACE_FILE, "rt", encoding="utf-8") as file:
    trace_root = ET.parse(file).getroot()


# --------------------------------------------------
# 2. Rasterize each path separately
# --------------------------------------------------

rasterized_segments = []
trace_path_count = 0
trace_point_count = 0

for path_node in trace_root.findall(".//path"):
    # Use voxel coordinates x, y, z.
    # Physical coordinates xd, yd, zd are not used.
    points_xyz = np.asarray(
        [
            [
                float(point.attrib["x"]),
                float(point.attrib["y"]),
                float(point.attrib["z"]),
            ]
            for point in path_node.findall("./point")
        ],
        dtype=np.float64,
    )

    if len(points_xyz) == 0:
        continue

    if not np.isfinite(points_xyz).all():
        raise ValueError("The tracing contains non-finite coordinates.")

    trace_path_count += 1
    trace_point_count += len(points_xyz)

    if len(points_xyz) == 1:
        rasterized_segments.append(
            np.rint(points_xyz).astype(np.int64)
        )
        continue

    for start, end in zip(points_xyz[:-1], points_xyz[1:]):
        coordinates = line_nd(
            np.rint(start).astype(np.int64),
            np.rint(end).astype(np.int64),
            endpoint=True,
        )

        rasterized_segments.append(
            np.column_stack(coordinates)
        )

if not rasterized_segments:
    raise ValueError("No tracing coordinates were found.")

trace_global_xyz = np.unique(
    np.concatenate(rasterized_segments, axis=0),
    axis=0,
)


# --------------------------------------------------
# 3. Convert to local ROI coordinates
# --------------------------------------------------

trace_local_xyz = trace_global_xyz.copy()

trace_local_xyz[:, 0] -= roi_x0
trace_local_xyz[:, 1] -= roi_y0

# The P28 ROI retains all Z slices, so Z is unchanged.
trace_x, trace_y, trace_z = trace_local_xyz.T
nz, ny, nx = shape_zyx

inside_roi = (
    (trace_x >= 0) & (trace_x < nx)
    & (trace_y >= 0) & (trace_y < ny)
    & (trace_z >= 0) & (trace_z < nz)
)

# Include all traced voxels within the ROI, regardless of activity.
trace_inside_xyz = trace_local_xyz[inside_roi]

if len(trace_inside_xyz) == 0:
    raise ValueError("No traced voxels fall inside the ROI.")


# --------------------------------------------------
# 4. Save the reference coordinates and their summary
# --------------------------------------------------

np.save(
    OUTPUT_DIR / "manual_trace_voxels_roi_xyz.npy",
    trace_inside_xyz,
)

trace_info = {
    "source_file": project_relative(TRACE_FILE),
    "coordinate_columns": "X,Y,Z",
    "coordinate_origin": "zero-based local ROI",
    "rasterization": "round endpoints, then line_nd with endpoint=True",
    "path_count": trace_path_count,
    "control_point_count": trace_point_count,
    "unique_voxels_full_trace": int(len(trace_global_xyz)),
    "unique_voxels_inside_roi": int(len(trace_inside_xyz)),
    "unique_voxels_outside_roi": int((~inside_roi).sum()),
    "roi_x0": roi_x0,
    "roi_y0": roi_y0,
    "roi_z0": 0,
}

(OUTPUT_DIR / "manual_trace_info.json").write_text(
    json.dumps(trace_info, indent=2),
    encoding="utf-8",
)

print(f"Tracing paths: {trace_path_count:,}")
print(f"Tracing control points: {trace_point_count:,}")
print(f"Unique traced voxels: {len(trace_global_xyz):,}")
print(f"Traced voxels inside ROI: {len(trace_inside_xyz):,}")
print(f"Traced voxels outside ROI: {(~inside_roi).sum():,}")

Tracing paths: 125
Tracing control points: 2,884
Unique traced voxels: 2,753
Traced voxels inside ROI: 2,491
Traced voxels outside ROI: 262


In [4]:
# Cell 4 — Calculate manual-tracing coverage

# --------------------------------------------------
# 1. Exact 3D coverage within the ROI
# --------------------------------------------------

trace_x, trace_y, trace_z = trace_inside_xyz.T

total_3d = int(len(trace_inside_xyz))

if total_3d == 0:
    raise ValueError("No reference voxels are available inside the ROI.")

# Coordinate columns are X,Y,Z; mask indexing is Z,Y,X.
covered_3d = final_mask_zyx[
    trace_z,
    trace_y,
    trace_x,
]

matched_3d = int(covered_3d.sum())
missed_3d = total_3d - matched_3d

coverage_3d_percent = 100.0 * matched_3d / total_3d


# --------------------------------------------------
# 2. Exact XY-projected coverage within the ROI
# --------------------------------------------------

# Deduplicate again because different Z locations may share X,Y.
trace_inside_xy = np.unique(
    trace_inside_xyz[:, :2],
    axis=0,
)

final_projection_yx = final_mask_zyx.any(axis=0)

covered_xy = final_projection_yx[
    trace_inside_xy[:, 1],
    trace_inside_xy[:, 0],
]

total_xy = int(len(trace_inside_xy))
matched_xy = int(covered_xy.sum())
missed_xy = total_xy - matched_xy

coverage_xy_percent = 100.0 * matched_xy / total_xy


# --------------------------------------------------
# 3. Build and save the coverage table
# --------------------------------------------------

coverage_summary = pd.DataFrame([
    {
        "evaluation": "3D exact, within ROI",
        "manual_locations": total_3d,
        "covered_locations": matched_3d,
        "missed_locations": missed_3d,
        "coverage_percent": coverage_3d_percent,
    },
    {
        "evaluation": "XY projected exact, within ROI",
        "manual_locations": total_xy,
        "covered_locations": matched_xy,
        "missed_locations": missed_xy,
        "coverage_percent": coverage_xy_percent,
    },
])

coverage_summary.to_csv(
    OUTPUT_DIR / "coverage_summary.csv",
    index=False,
)


# --------------------------------------------------
# 4. Save the evaluation scope and counts
# --------------------------------------------------

coverage_info = {
    "sample": SAMPLE_NAME,
    "primary_metric": "exact 3D coverage within ROI",
    "reference_denominator": "all unique traced voxels inside ROI",
    "traced_3d_voxels": total_3d,
    "covered_3d_voxels": matched_3d,
    "missed_3d_voxels": missed_3d,
    "coverage_3d_percent": coverage_3d_percent,
    "traced_xy_pixels": total_xy,
    "covered_xy_pixels": matched_xy,
    "coverage_xy_percent": coverage_xy_percent,
    "traced_voxels_outside_roi": int((~inside_roi).sum()),
}

(OUTPUT_DIR / "coverage_info.json").write_text(
    json.dumps(coverage_info, indent=2),
    encoding="utf-8",
)

print(
    coverage_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
    )
)

print(
    "\nTracing voxels outside the ROI "
    f"(excluded from these denominators): {(~inside_roi).sum():,}"
)
print(f"Saved results to: {project_relative(OUTPUT_DIR)}")

                    evaluation  manual_locations  covered_locations  missed_locations  coverage_percent
          3D exact, within ROI              2491               2092               399             83.98
XY projected exact, within ROI              2399               2099               300             87.49

Tracing voxels outside the ROI (excluded from these denominators): 262
Saved results to: results/p28_330_HET_009/08_evaluation


In [5]:
# Cell 5 — Compare the three path-length correction methods

# --------------------------------------------------
# 1. Calculate the inverse-L score
# --------------------------------------------------

minimum_length = float(
    run_info["path_length"]["minimum_effective_length"]
)

valid_zyx = (
    active_zyx
    & np.isfinite(L_zyx)
    & (L_zyx >= minimum_length)
    & (P_hit_zyx > 0)
)

score_inverse_L_zyx = np.zeros(
    shape_zyx,
    dtype=np.float32,
)

score_inverse_L_zyx[valid_zyx] = np.exp(
    np.log(P_hit_zyx[valid_zyx].astype(np.float64))
    / L_zyx[valid_zyx].astype(np.float64)
).astype(np.float32)

score_inverse_L_zyx[target_zyx] = 1.0

# The uncorrected and inverse-sqrt(L) scores were saved in Step07.
score_versions = [
    ("uncorrected", "P_hit", P_hit_zyx),
    ("inverse_L", "P_hit ** (1 / L)", score_inverse_L_zyx),
    ("inverse_sqrtL", "P_hit ** (1 / sqrt(L))", sqrtL_score_zyx),
]


# --------------------------------------------------
# 2. Define identical mask filtering for all methods
# --------------------------------------------------

raw_gate_yx = normalized_intensity_yx > INTENSITY_THRESHOLD


def make_comparison_mask(score_zyx):
    candidate = (
        active_zyx
        & (score_zyx > SCORE_THRESHOLD)
        & raw_gate_yx[None, :, :]
    )

    candidate |= target_zyx

    if not APPLY_COMPONENT_FILTER:
        return candidate

    labels, _ = ndi.label(
        candidate,
        structure=CONNECTIVITY_3D,
    )

    sizes = np.bincount(labels.ravel())
    keep = sizes >= MIN_COMPONENT_VOXELS
    keep[0] = False

    # Retain entire components containing soma-target voxels.
    soma_ids = np.unique(labels[target_zyx])
    soma_ids = soma_ids[soma_ids > 0]
    keep[soma_ids] = True

    mask = keep[labels]
    mask |= target_zyx

    return mask


# --------------------------------------------------
# 3. Calculate coverage for each method
# --------------------------------------------------

correction_comparison_masks = {}
comparison_rows = []

for name, formula, score_zyx in score_versions:
    mask_zyx = make_comparison_mask(score_zyx)
    mask_yx = mask_zyx.any(axis=0)

    matched_3d = int(
        mask_zyx[trace_z, trace_y, trace_x].sum()
    )

    matched_xy = int(
        mask_yx[
            trace_inside_xy[:, 1],
            trace_inside_xy[:, 0],
        ].sum()
    )

    correction_comparison_masks[name] = mask_zyx

    comparison_rows.append({
        "version": name,
        "score_formula": formula,
        "traced_3d_voxels": total_3d,
        "covered_3d_voxels": matched_3d,
        "missed_3d_voxels": total_3d - matched_3d,
        "coverage_3d_percent": 100.0 * matched_3d / total_3d,
        "traced_xy_pixels": total_xy,
        "covered_xy_pixels": matched_xy,
        "coverage_xy_percent": 100.0 * matched_xy / total_xy,
        "selected_3d_voxels": int(mask_zyx.sum()),
        "selected_xy_pixels": int(mask_yx.sum()),
        "score_threshold": SCORE_THRESHOLD,
        "raw_mip_intensity_threshold": INTENSITY_THRESHOLD,
        "component_filtering": APPLY_COMPONENT_FILTER,
        "minimum_component_voxels": MIN_COMPONENT_VOXELS,
    })

correction_comparison_table = pd.DataFrame(comparison_rows)


# --------------------------------------------------
# 4. Verify that the baseline reproduces the saved final mask
# --------------------------------------------------

if not np.array_equal(
    correction_comparison_masks["inverse_sqrtL"],
    final_mask_zyx,
):
    raise RuntimeError(
        "The reconstructed inverse-sqrt(L) mask differs from "
        "the saved Step07 final mask. Check the filtering settings."
    )


# --------------------------------------------------
# 5. Save the comparison results
# --------------------------------------------------

correction_comparison_table.to_csv(
    OUTPUT_DIR / "correction_comparison.csv",
    index=False,
)

np.save(
    OUTPUT_DIR / "score_inverse_L_zyx.npy",
    score_inverse_L_zyx,
)

for name, mask_zyx in correction_comparison_masks.items():
    np.save(
        OUTPUT_DIR / f"mask_{name}_zyx.npy",
        mask_zyx,
    )

print(
    correction_comparison_table[
        [
            "version",
            "coverage_3d_percent",
            "coverage_xy_percent",
            "selected_3d_voxels",
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.2f}",
    )
)

print(f"\nSaved comparison results to: {project_relative(OUTPUT_DIR)}")

      version  coverage_3d_percent  coverage_xy_percent  selected_3d_voxels
  uncorrected                35.33                37.31               81621
    inverse_L                90.57                90.29              441521
inverse_sqrtL                83.98                87.49              233435

Saved comparison results to: results/p28_330_HET_009/08_evaluation


In [ ]:
# Cell 6 — Save evaluation provenance and the output manifest
#
# Preserve the reconstruction record and evaluation settings for Step09.
# Directory and tracing paths are relative to the repository root;
# output filenames are relative to this evaluation directory.


# --------------------------------------------------
# 1. Preserve the reconstruction record and ROI metadata
# --------------------------------------------------

source_run_info_file = (
    OUTPUT_DIR / "source_reconstruction_run_info.json"
)

source_run_info_file.write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8",
)

metadata_table.to_csv(
    OUTPUT_DIR / "roi_metadata.csv",
    index=False,
)


# --------------------------------------------------
# 2. Check the evaluation files produced by Cells 3–5
# --------------------------------------------------

evaluation_files = {
    "source_run_info": "source_reconstruction_run_info.json",
    "roi_metadata": "roi_metadata.csv",
    "manual_trace_voxels": "manual_trace_voxels_roi_xyz.npy",
    "manual_trace_info": "manual_trace_info.json",
    "coverage_table": "coverage_summary.csv",
    "coverage_info": "coverage_info.json",
    "correction_comparison": "correction_comparison.csv",
    "inverse_L_score": "score_inverse_L_zyx.npy",
}

correction_mask_files = {
    name: f"mask_{name}_zyx.npy"
    for name in correction_comparison_masks
}

missing_files = [
    project_relative(OUTPUT_DIR / filename)
    for filename in [
        *evaluation_files.values(),
        *correction_mask_files.values(),
    ]
    if not (OUTPUT_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        "Missing evaluation files. Complete Cells 1–5 first:\n"
        + "\n".join(missing_files)
    )


# --------------------------------------------------
# 3. Record the sources, evaluation scope, and matched filtering
# --------------------------------------------------

evaluation_info = {
    "sample": SAMPLE_NAME,
    "path_base": "repository_root",
    "reconstruction_directory": project_relative(RECONSTRUCTION_DIR),
    "evaluation_directory": project_relative(OUTPUT_DIR),
    "manual_trace_source": project_relative(TRACE_FILE),
    "array_order_3d": run_info["array_order_3d"],
    "trace_coordinate_columns": trace_info["coordinate_columns"],
    "shape_zyx": [int(n) for n in shape_zyx],
    "evaluation_scope": "manual-traced voxels within the ROI",
    "primary_metric": coverage_info["primary_metric"],
    "secondary_metric": "exact XY-projected coverage within ROI",
    "trace": trace_info,
    "coverage": coverage_info,
    "comparison_versions": [
        name for name, formula, score_zyx in score_versions
    ],
    "matched_filtering": {
        "score_threshold": float(SCORE_THRESHOLD),
        "raw_mip_intensity_threshold": float(INTENSITY_THRESHOLD),
        "component_filtering": bool(APPLY_COMPONENT_FILTER),
        "minimum_component_voxels": (
            int(MIN_COMPONENT_VOXELS)
            if APPLY_COMPONENT_FILTER
            else None
        ),
        "soma_targets_retained": True,
    },
    "files": evaluation_files,
    "correction_mask_files": correction_mask_files,
}


# --------------------------------------------------
# 4. Save the evaluation record
# --------------------------------------------------

evaluation_info_file = OUTPUT_DIR / "evaluation_info.json"

evaluation_info_file.write_text(
    json.dumps(evaluation_info, indent=2),
    encoding="utf-8",
)

print(f"Evaluation record: {project_relative(evaluation_info_file)}")
print(f"Results directory: {project_relative(OUTPUT_DIR)}")
